---
---
# __Table of Contents__


## __Data Cleaning & Fixing__

### __1. Library Imports__
* __Data Manipulation:__ `pandas` and `numpy` to structure and clean your numbers.

### __2. Structural Gaps (Handling Missing Data)__
1. Working on `train.csv` dataset
2. Working on `test.csv` dataset 
---
---

### __Data Dictionary__

|Variable	     |Definition	     |Key                                             |
|:---------------|:------------------|:-----------------------------------------------|
|passenger_id    |Passenger ID       |
|survival  	     |Survival	         |0 = No, 1 = Yes                                 |
|ticket_class    |Ticket class       |1 = 1st, 2 = 2nd, 3 = 3rd                       |
|name            |Name of person     |
|sex	         |Sex	             |
|age	         |Age in years       |	
|siblings_spouses|# of siblings / spouses aboard the Titanic|
|parents_children|# of parents / children aboard the Titanic|
|ticket	         |Ticket number      |
|fare	         |Passenger fare     |	
|cabin	         |Cabin number       |	
|boarding_port   |Port of Embarkation|	C = Cherbourg, Q = Queenstown, S = Southampton|


### __Variable Notes__

__ticket_class:__ A proxy for socio-economic status (SES)
* 1st = Upper
* 2nd = Middle
* 3rd = Lower

__age:__ Age is fractional if less than 1. If the age is estimated, is it in the form of xx.5

__siblings_spouses:__ The dataset defines family relations in this way...
* Sibling = brother, sister, stepbrother, stepsister
* Spouse = husband, wife (mistresses and fiancés were ignored)

__parents_children:__ The dataset defines family relations in this way...
* Parent = mother, father
* Child = daughter, son, stepdaughter, stepson
* Some children travelled only with a nanny, therefore parch=0 for them.

### __1. Library Imports__
---

In [1]:
# Data manipulation and calculations
import pandas as pd
import numpy as np

### __2. Structural Gaps (Handling Missing Data)__
---

### __2.1 Working On `train.csv` Dataset__

#### __Stage 1 -->__

#### __Load the Dataset__

In [2]:
# Import the 'train.csv' dataset from 2-under-process folder
train_df = pd.read_csv('../data/2-under-process/train.csv')

In [3]:
# Show the first 5 rows of the dataset
train_df.head()

,passenger_id,survival,ticket_class,name,sex,age,siblings_spouses,parents_children,ticket,fare,cabin,boarding_port
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Get the structural breakdown of the dataset
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   passenger_id      891 non-null    int64  
 1   survival          891 non-null    int64  
 2   ticket_class      891 non-null    int64  
 3   name              891 non-null    object 
 4   sex               891 non-null    object 
 5   age               714 non-null    float64
 6   siblings_spouses  891 non-null    int64  
 7   parents_children  891 non-null    int64  
 8   ticket            891 non-null    object 
 9   fare              891 non-null    float64
 10  cabin             204 non-null    object 
 11  boarding_port     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [5]:
# Create a quick summary table of missing values
missing_counts = train_df.isnull().sum()
missing_percentages = (train_df.isnull().sum() / len(train_df)) * 100

# Combine them into a clean visualization dataframe
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percentages.round(2)
})

# Filter out columns that have 0 missing values to keep it punchy
missing_summary[missing_summary['Missing Count'] > 0]

,Missing Count,Percentage (%)
age,177,19.87
cabin,687,77.10
boarding_port,2,0.22


#### 📝 __Analysis Note: _Missing Data Strategy___
* __`cabin` (77.10% Missing):__ _Ignore and set aside._ Because it exceeds our _60% critical threshold,_ trying to fill these gaps directly would introduce massive statistical bias and noise. We will leave it as an `"Unknown"` text placeholder for now and extract its hidden patterns (like Deck Levels) during Feature Engineering later.

* __`boarding_port` (0.22% Missing) & `age` (19.87% Missing):__ _Keep and fix._ These columns fall well within the fixable range and hold vital survival clues, so we will focus our next steps on programmatically repairing their gaps.

In [6]:
# Safely replace all NaN values with the string "Unknown"
train_df['cabin'] = train_df['cabin'].fillna("Unknown")

# Quick verification check using pandas
f"Remaining missing values in the 'cabin' column: {train_df['cabin'].isnull().sum()}"

"Remaining missing values in the 'cabin' column: 0"

In [7]:
# Check missing values of 'boarding_port' column
train_df[train_df['boarding_port'].isnull() == True]

,passenger_id,survival,ticket_class,name,sex,age,siblings_spouses,parents_children,ticket,fare,cabin,boarding_port
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN


In [8]:
# Check if anyone else was in the B28 cabin
f"Number of People in the 'B28' Cabin: {len(train_df[train_df['cabin'] == 'B28'])}"

"Number of People in the 'B28' Cabin: 2"

In [9]:
# 1. Fill the missing values of 'boarding_port' column
# 1.1 Calculate the mode for 1st class passengers cleanly
port_mode_1st = train_df[train_df['ticket_class'] == 1]['boarding_port'].mode()[0]

# 1.2 Fill the missing values safely by reassigning the column
train_df['boarding_port'] = train_df['boarding_port'].fillna(port_mode_1st)

# 1.3 Print the final audit report
print(f"Remaining missing values in the 'boarding_port' column: {train_df['boarding_port'].isnull().sum()}")

Remaining missing values in the 'boarding_port' column: 0


#### __Save the Changes__

In [10]:
# Save the data into under-process folder
train_df.to_csv('../data/2-under-process/train.csv', index=False)